# VoxCeleb2: acquiring a corpus with more than one sighting per person

`06_facial` is limited by **sessions, not frames**. Measured on its own validation split:

| videos available | identities | rank-1 |
| --- | --- | --- |
| 1 | 45 | 26.7% |
| 2 | 29 | 58.6% |
| 3 | 15 | 60.0% |
| 4 | 8 | 75.0% |
| 5 | 3 | 100.0% |

Monotonic, +37 points between one video and two or more, about four standard errors. Frames of
one YouTube Faces video sit at 0.78 mean cosine against 0.42 across videos, and a video is only
~3 seconds — so eight frames of one clip are one observation, eight times over. Once a target's
video is held out, 47% of identities have exactly one left.

VoxCeleb2 has roughly **24 videos per identity over ~6,000 identities**, against an effective 1.8
here. That is the axis the premise lives on.

### Where it comes from, and why that needs saying

The VGG group no longer distributes it — their page states "The video files for the VoxCeleb 2
dataset are no longer available from this website", and likewise for the audio, the URLs and the
identifying metadata. This notebook pulls from the
[`ProgramComputer/voxceleb`](https://huggingface.co/datasets/ProgramComputer/voxceleb) mirror,
which is a **third-party re-upload of data the original curators withdrew**. That page is candid
about it:

> "This dataset contains images or recordings of real, identifiable people that third parties
> collected from the web." … "access here is not a licence to it."

For a thesis arguing that anonymization is defeatable, using a withdrawn corpus of identifiable
people is a decision to make deliberately and to state in the data section — not one to slide
past. It is worth clearing with your supervisor or ethics review before the download, not after.

## 0. Setup

The dataset is **gated**: open the dataset page, accept the conditions, then authenticate with
`hf auth login` (or set `HF_TOKEN`). The cell below fails early and says so if that has not
happened, rather than part way through a 50 GiB transfer.

Two things make this download cheaper than it looks:

- `allow_patterns` / explicit filenames mean VoxCeleb1 and the ~80 GiB of `aac` audio are never
  touched. Only `vox2_mp4_*.zip` and the metadata are fetched.
- `local_dir` puts the files straight on the data partition. The default cache lives under `~`,
  which would both fill a smaller disk and store every byte twice.

Downloads resume, so an interrupted transfer costs only what it had not finished.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import re
import shutil
import zipfile

# Xet's high-performance path, worth having on archives this size. Set before huggingface_hub is
# imported, because the flag is read at import time. (The older HF_HUB_ENABLE_HF_TRANSFER and its
# hf_transfer backend were removed in huggingface_hub 2.0 and now only warn.)
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")

from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.errors import GatedRepoError
from tqdm.auto import tqdm

from meles.dataset import extract_video_frames, read_dataset
from meles.embedding_aggregation import normalized
from meles.frame_embedder import FrameEmbedder, models

REPO = "ProgramComputer/voxceleb"
DATA_PATH = Path("../../data")
DOWNLOADS = DATA_PATH / "vox2_download"  # the archives, as fetched
STAGING = DATA_PATH / "vox2_clips"  # mp4s for the identities we keep
FRAMES = DATA_PATH / "voxceleb"  # identity/video/frame.jpg, what the pipeline reads

# All six, processed one at a time. Each is handled start to finish before the next is
# touched, so an interrupted run resumes at archive granularity rather than starting over.
ARCHIVES = [f"vox2/vox2_mp4_{n}.zip" for n in range(1, 7)]
FRAMES_PER_VIDEO = 25  # spread across the video's clips, never consecutive
WORKERS = None  # extraction processes; None means every core but one
EXTRACTED = DOWNLOADS / "extracted"  # one marker file per finished archive, for resuming

# Listing is public even though the files are not, so it cannot stand in for an access check --
# the gate only bites on download. The real check is step 1, which fetches 159 kB of metadata.
repo_files = HfApi().list_repo_files(REPO, repo_type="dataset")
print(f"{len(repo_files)} files in the repo. The mp4 archives and metadata:")
for name in sorted(repo_files):
    if name.startswith("vox2/") and ("mp4" in name or "meta" in name):
        print(f"  {name}")

/data/projects/personal/meles/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


41 files in the repo. The mp4 archives and metadata:
  vox2/vox2_dev_mp4_partaa
  vox2/vox2_dev_mp4_partab
  vox2/vox2_dev_mp4_partac
  vox2/vox2_dev_mp4_partad
  vox2/vox2_dev_mp4_partae
  vox2/vox2_dev_mp4_partaf
  vox2/vox2_dev_mp4_partag
  vox2/vox2_dev_mp4_partah
  vox2/vox2_dev_mp4_partai
  vox2/vox2_meta.csv
  vox2/vox2_mp4_1.zip
  vox2/vox2_mp4_2.zip
  vox2/vox2_mp4_3.zip
  vox2/vox2_mp4_4.zip
  vox2/vox2_mp4_5.zip
  vox2/vox2_mp4_6.zip
  vox2/vox2_test_mp4.zip


## 1. The metadata first

159 kB, and it is also the access check — it fails exactly the way a 50 GiB archive would.

It carries less than you might hope. VoxCeleb**1**'s metadata names its people
(`id10001 → A.J._Buckley`); VoxCeleb2's does not:

```
VoxCeleb2 ID ,VGGFace2 ID ,Gender ,Set
id00012 ,n000012 ,m ,dev
```

Comma separated, space padded, UTF-8 BOM, and **no name column** — the identifying metadata is
part of what VGG withdrew. The `VGGFace2 ID` would resolve through VGGFace2's `identity_meta.csv`,
but that is a second gated re-upload of a second withdrawn corpus, which is a price worth not
paying for what turns out to be the weaker check anyway (step 2).

So identities stay as `id00012` throughout. Nothing downstream needs a name: the gallery is
YouTube Faces and is scored separately, so the directory label is cosmetic.

In [2]:
# Also the access check: 159 kB, and it fails the same way a 50 GiB archive would.
try:
    meta_path = hf_hub_download(REPO, "vox2/vox2_meta.csv", repo_type="dataset", local_dir=DOWNLOADS)
except GatedRepoError as error:
    raise RuntimeError(
        f"No access to {REPO}. The repo is gated, and listing its files is public while "
        "downloading them is not, so this is the first place it shows up.\n"
        f"  1. open https://huggingface.co/datasets/{REPO} and accept the conditions "
        "(granted immediately, no manual review)\n"
        "  2. run 'hf auth login', or set HF_TOKEN to a token from "
        "https://huggingface.co/settings/tokens"
    ) from error

# Comma separated with the fields padded by spaces, and a BOM on the header -- strip all three,
# or the first column comes back named '﻿VoxCeleb2 ID' and every id carries a trailing space.
meta = pd.read_csv(meta_path, sep=",", encoding="utf-8-sig")
meta.columns = [c.strip() for c in meta.columns]
for column in meta.columns:
    if meta[column].dtype == object:
        meta[column] = meta[column].str.strip()

meta = meta.rename(columns={"VoxCeleb2 ID": "vox_id", "Set": "split"})
print(f"columns: {list(meta.columns)}")
print(f"{len(meta):,} identities: {(meta['split'] == 'dev').sum():,} dev, {(meta['split'] == 'test').sum():,} test")
meta.head()

columns: ['vox_id', 'VGGFace2 ID', 'Gender', 'split']
6,114 identities: 5,994 dev, 120 test


,vox_id,VGGFace2 ID,Gender,split
0,id00012,n000012,m,dev
1,id00015,n000015,m,dev
2,id00016,n000016,m,dev
3,id00017,n000017,m,test
4,id00018,n000018,m,dev


## 2. Which identities, and why the gallery check is deferred

`meles.training.pairs` trains pix2pix on a different corpus on purpose, so that "the generator's
prior [does not] contain the gallery". `Facial` has been violating that by training on the very
dataset the experiment scores. VoxCeleb2 fixes it — but only if the people do not overlap, and
both corpora are scraped celebrities, so some will.

The plan was a set intersection on names. Without names that is not available, and it is worth
saying that this is **not purely a loss**. Run the embedding check over YouTube Faces against
itself and it finds three pairs of identities that are the same person under two labels:

| cosine | label A | label B | |
| --- | --- | --- | --- |
| 0.981 | `Odai_Hussein` | `Uday_Hussein` | transliteration |
| 0.795 | `Felipe_De_Borbon` | `Prince_Felipe` | title vs. name |
| 0.716 | `Cristina_Fernandez` | `Cristina_Kirchner` | maiden vs. married |
| 0.716 | `Ali_Mohammed_Maher` | `Gary_Condit` | **not** the same person |

A string comparison misses the first three, so it was never going to deliver the "provably
disjoint" claim it looked like it would — it compares spellings, and the thing that has to be
disjoint is *people*. Step 5 compares embeddings instead, which is both the stronger claim and
the one phrased in terms of what the recogniser actually sees.

The fourth row is the honest half of the same picture: two different men at the same 0.716 as a
true pair, which is why the threshold below is calibrated rather than guessed, and why it is set
to err towards dropping.

That check needs frames, so it runs after extraction. Here we only drop the 120 `test`
identities, which the VoxCeleb2 protocol holds out anyway.

In [3]:
ytfaces = pd.read_parquet(DATA_PATH / "ytfaces_wide.parquet")
gallery_identities = ytfaces["identity"].nunique()

# Only the dev split; the gallery-overlap filter happens in step 5, once there are faces to embed.
wanted_ids = set(meta.loc[meta["split"] == "dev", "vox_id"])

print(f"YouTube Faces gallery: {gallery_identities:,} identities, {len(ytfaces):,} frames")
print(f"VoxCeleb2 dev:         {len(wanted_ids):,} identities  (test split held out: "
      f"{(meta['split'] == 'test').sum():,})")
print(f"\nidentities are kept as ids, e.g. {sorted(wanted_ids)[:3]}")

YouTube Faces gallery: 1,004 identities, 195,851 frames
VoxCeleb2 dev:         5,994 identities  (test split held out: 120)

identities are kept as ids, e.g. ['id00012', 'id00015', 'id00016']


## 3. Fetch, stage and extract, one archive at a time

267 GB over six archives, and the cost is lopsided: on this machine archive 1 measured 86 min to
download but **157 min to extract**. Three things follow.

**Download it outside the notebook.** The transfer needs no kernel and resumes, so start it in a
terminal and let it run overnight:

```shell
hf download ProgramComputer/voxceleb --repo-type dataset \
   --include "vox2/vox2_mp4_*.zip" --local-dir data/vox2_download
```

The loop below then finds every archive already cached and goes straight to extracting.

**Staging is cleared between archives.** `extract_video_frames` walks whatever is in `STAGING`,
so letting six archives pile up there would make archive 6 reprocess the five before it — and
hold ~270 GB instead of ~50 GB.

**Each archive is marked done when its frames are written.** Re-running skips finished archives,
so a dead kernel costs the archive in flight and nothing else. To force a redo — a different
`FRAMES_PER_VIDEO`, say — delete `data/vox2_download/extracted/`.

The probe is inside the loop now: `vox2_mp4_N.zip` *should* each open standalone, and `BadZipFile`
is how we would find out otherwise, per archive rather than after all six.

In [4]:
IDENTITY = re.compile(r"(id\d{5})")


def identity_of(member):
    found = IDENTITY.search(member)
    return found.group(1) if found else None


DOWNLOADS.mkdir(parents=True, exist_ok=True)
EXTRACTED.mkdir(parents=True, exist_ok=True)

for entry in ARCHIVES:
    marker = EXTRACTED / (Path(entry).name + ".done")
    if marker.exists():
        print(f"{Path(entry).name}: {marker.read_text().strip()} -- skipping")
        continue

    print(f"\n=== {Path(entry).name} ===", flush=True)
    archive_path = Path(hf_hub_download(REPO, entry, repo_type="dataset", local_dir=DOWNLOADS))
    print(f"  {archive_path.stat().st_size / 1e9:.1f} GB on disk", flush=True)

    try:
        with zipfile.ZipFile(archive_path) as archive:
            members = [n for n in archive.namelist() if n.lower().endswith(".mp4")]
    except zipfile.BadZipFile:
        raise RuntimeError(
            f"{archive_path.name} is not a standalone zip, so it is one volume of a set. Download "
            "every vox2_mp4_*.zip (or the vox2_dev_mp4_part* files and 'cat' them) first."
        ) from None

    usable = {identity_of(n) for n in members} & wanted_ids
    print(f"  {len(members):,} clips, {len(usable):,} dev identities", flush=True)

    # Cleared every round: extraction must see this archive's clips and nothing else.
    shutil.rmtree(STAGING, ignore_errors=True)
    STAGING.mkdir(parents=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in tqdm([n for n in members if identity_of(n) in usable], desc="  staging", mininterval=5.0):
            archive.extract(member, STAGING)

    # The archive nests the ids under something like 'dev/mp4/'; find whatever holds them.
    roots = {d.parent for d in STAGING.rglob("id?????") if d.is_dir()}
    part = extract_video_frames(
        sorted(roots)[0] if roots else STAGING,
        FRAMES,
        frames_per_video=FRAMES_PER_VIDEO,
        require_face=True,  # a frame with no face is not an observation of anybody
        workers=WORKERS,
    )
    marker.write_text(f"{len(part):,} frames over {part['identity'].nunique():,} identities")
    print(f"  {marker.read_text()}", flush=True)

shutil.rmtree(STAGING, ignore_errors=True)
print("\nall archives extracted, staging cleared")

vox2_mp4_1.zip: 652,395 frames over 1,118 identities (run of 2026-10-26; 39 later dropped by the gallery filter) -- skipping

=== vox2_mp4_2.zip ===
  50.0 GB on disk
  204,726 clips, 1,117 dev identities


extracting:   3%|▎         | 36/1117 [01:31<39:46,  2.21s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
extracting:  30%|██▉       | 330/1117 [17:37<39:06,  2.98s/it]Could not open clip
Could not open clip
extracting:  49%|████▉     | 545/1117 [28:07<29:20,  3.08s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
extracting:  54%|█████▍    | 605/1117 [31:19<24:59,  2.93s/it]Could not open clip
Could not open clip
Could not open clip
extracting:  95%|█████████▍| 1060/1117 [54:33<03:39,  3.85s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
extracting: 100%|██████████| 1117/1117 [56:43<00:00,  3.05s/it]


  658,819 frames over 1,117 identities

=== vox2_mp4_3.zip ===
  50.0 GB on disk
  201,705 clips, 1,098 dev identities


extracting:  42%|████▏     | 466/1098 [21:40<24:48,  2.36s/it]Could not open clip
Could not open clip
Could not open clip
extracting:  43%|████▎     | 469/1098 [21:47<25:10,  2.40s/it]Could not open clip
Could not open clip
Could not open clip
extracting:  57%|█████▋    | 625/1098 [29:36<25:51,  3.28s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
extracting:  57%|█████▋    | 627/1098 [29:46<28:30,  3.63s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
extracting:  83%|████████▎ | 908/1098 [42:37<10:36,  3.35s/it][mov,mp4,m4a,3gp,3g2,mj2 @ 0xaadf280] moov atom not found
Could not open clip
extracting:  93%|█████████▎| 1022/1098 [47:56<03:31,  2.78s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not

  647,085 frames over 1,098 identities

=== vox2_mp4_4.zip ===
  50.0 GB on disk
  205,178 clips, 1,156 dev identities


...

  671,984 frames over 1,156 identities

=== vox2_mp4_5.zip ===
  50.0 GB on disk
  205,826 clips, 1,152 dev identities


extracting:  51%|█████     | 585/1152 [28:17<28:05,  2.97s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
extracting:  51%|█████     | 587/1152 [28:30<35:48,  3.80s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
extracting:  63%|██████▎   | 726/1152 [35:36<23:09,  3.26s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
extracting:  63%|██████▎   | 728/1152 [35:45<25:29,  3.61s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not op

  658,194 frames over 1,152 identities

=== vox2_mp4_6.zip ===
  17.0 GB on disk
  67,758 clips, 358 dev identities


extracting:  86%|████████▌ | 307/358 [15:26<02:45,  3.24s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
extracting:  86%|████████▌ | 308/358 [15:37<03:36,  4.34s/it]Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
Could not open clip
extracting:  87%|████████▋ | 312/358 [15:45<02:30,  3.28s/it]Could not open clip
Could not open clip
extracting: 100%|██████████| 358/358 [17:59<00:00,  3.01s/it]


  212,599 frames over 358 identities

all archives extracted, staging cleared


## 4. The structure

Read back from disk rather than stitched together from what the loop returned. That makes it
independent of how many archives this particular run happened to process — a resumed run and a
clean run produce the same thing — and it is the same `read_dataset` call `02_ytfaces_dataset`
uses, so the two corpora are described by exactly one piece of code.

In [5]:
structure = read_dataset(FRAMES, root=DATA_PATH)

per_identity = structure.groupby("identity")["video"].nunique()
print(f"{len(structure):,} frames over {structure['identity'].nunique():,} identities")
print(f"videos per identity: min {per_identity.min()}, median {int(per_identity.median())}, "
      f"max {per_identity.max()}, mean {per_identity.mean():.1f}")
print("  (YouTube Faces, after holding out the target's video: 1.8)")
structure.head()

3,472,477 frames over 5,955 identities
videos per identity: min 6, median 20, max 91, mean 24.2
  (YouTube Faces, after holding out the target's video: 1.8)


,identity,video,frame,frame_suffix,path
0,id06595,dcxzmpDDOgo,00017.0015.jpg,.jpg,voxceleb/id06595/dcxzmpDDOgo/00017.0015.jpg
1,id06595,dcxzmpDDOgo,00021.0022.jpg,.jpg,voxceleb/id06595/dcxzmpDDOgo/00021.0022.jpg
2,id06595,dcxzmpDDOgo,00010.0001.jpg,.jpg,voxceleb/id06595/dcxzmpDDOgo/00010.0001.jpg
3,id06595,dcxzmpDDOgo,00018.0016.jpg,.jpg,voxceleb/id06595/dcxzmpDDOgo/00018.0016.jpg
4,id06595,dcxzmpDDOgo,00018.0017.jpg,.jpg,voxceleb/id06595/dcxzmpDDOgo/00018.0017.jpg


## 5. Make the corpus disjoint from the gallery

Two steps, because a threshold asserted is worth nothing and a threshold measured is worth a line
in the thesis.

**Calibrate on the gallery itself.** Split each YouTube Faces identity's videos into two disjoint
halves and take the cosine between the two centroids: that is the same comparison the check makes
across corpora — one person, two separate sightings, never the same footage twice. Against it,
the best-matching *other* identity gives the impostor side. `TAU` is then read off the genuine
distribution at the recall we want rather than picked by eye.

Erring loose is the cheap direction: dropping an extra 2% of 5,994 identities costs ~120 people
out of thousands, while keeping one person who is also in the gallery contaminates the claim the
whole experiment rests on.

**Then apply it.** 32 frames per VoxCeleb2 identity, centroid, cosine against all 1,004 gallery
centroids, drop anything at or above `TAU`. The frames of dropped identities are deleted too, so
the directory and the parquet cannot drift apart.

What this buys over the name check: it catches the same person under two spellings, which is not
hypothetical — it finds three such pairs inside the gallery alone. The limit is honest and worth
stating: at 90% genuine recall roughly one in ten true overlaps slips through, so the claim is
"no identity within cosine `TAU` of the gallery", not "provably disjoint".

In [6]:
GENUINE_RECALL = 0.90  # the share of true same-person pairs tau has to catch

embeddings = pd.read_parquet(DATA_PATH / "ytfaces_embeddings.parquet",
                             columns=["identity", "video", "embedding"])
unit = np.asarray(normalized(np.stack(embeddings["embedding"].apply(np.asarray).values)), dtype=np.float32)


def centroid(rows):
    """Mean of unit vectors, renormalised -- a direction, so it compares by plain cosine."""
    mean = rows.mean(axis=0)
    return mean / np.linalg.norm(mean)


gallery_names = np.array(sorted(embeddings["identity"].unique()))
identity_of = embeddings["identity"].values
video_of = embeddings["video"].values
gallery_centroids = np.stack([centroid(unit[identity_of == name]) for name in gallery_names])

# Genuine: one person seen in two disjoint sets of videos. This is the comparison the check will
# actually make across corpora -- different sightings, never the same footage twice.
rng = np.random.default_rng(0)
genuine = []
for name in gallery_names:
    rows = identity_of == name
    videos = np.unique(video_of[rows])
    if len(videos) < 2:
        continue
    rng.shuffle(videos)
    half = len(videos) // 2
    left = unit[rows & np.isin(video_of, videos[:half])]
    genuine.append(float(centroid(left) @ centroid(unit[rows & np.isin(video_of, videos[half:])])))
genuine = np.array(genuine)

# Impostor: each identity's best match among the others.
similarity = gallery_centroids @ gallery_centroids.T
np.fill_diagonal(similarity, -np.inf)
impostor = similarity.max(axis=1)

TAU = float(np.quantile(genuine, 1 - GENUINE_RECALL))
print(f"genuine (n={len(genuine)}):  median {np.median(genuine):.3f}, p10 {np.quantile(genuine, 0.1):.3f}")
print(f"impostor (n={len(impostor)}): median {np.median(impostor):.3f}, p99 {np.quantile(impostor, 0.99):.3f}")
print(f"\ntau = {TAU:.3f}  -- catches {(genuine >= TAU).mean():.1%} of true same-person pairs, "
      f"flags {(impostor >= TAU).mean():.1%} of unrelated ones")

# The same-person-twice labels inside the gallery, which is what the string check would have missed.
pairs = [(similarity[a, b], gallery_names[a], gallery_names[b])
         for a, b in zip(*np.triu_indices(len(gallery_names), k=1)) if similarity[a, b] >= 0.6]
print(f"\ngallery labels that are the same person ({len(pairs)} pairs above 0.60):")
for score, left_name, right_name in sorted(pairs, reverse=True):
    print(f"  {score:.3f}  {left_name:<26} {right_name}")

genuine (n=1004):  median 0.652, p10 0.355
impostor (n=1004): median 0.246, p99 0.468

tau = 0.355  -- catches 89.9% of true same-person pairs, flags 2.4% of unrelated ones

gallery labels that are the same person (4 pairs above 0.60):
  0.981  Odai_Hussein               Uday_Hussein
  0.795  Felipe_De_Borbon           Prince_Felipe
  0.716  Cristina_Fernandez         Cristina_Kirchner
  0.716  Ali_Mohammed_Maher         Gary_Condit


In [7]:
CHECK_FRAMES = 32  # per identity, shuffled first so they spread over that identity's videos

# Shuffle then head: a plain head() would take 32 frames of the first video and call it a person.
probe = structure.sample(frac=1, random_state=0).groupby("identity", sort=False).head(CHECK_FRAMES)
arcface = FrameEmbedder(models.ARCFACE)
probe_unit = np.asarray(
    normalized(np.stack(arcface.embed([str(DATA_PATH / path) for path in probe["path"]]))),
    dtype=np.float32,
)

vox_names = np.array(sorted(probe["identity"].unique()))
vox_of = probe["identity"].values
vox_centroids = np.stack([centroid(probe_unit[vox_of == name]) for name in vox_names])

# Every VoxCeleb2 identity against every gallery identity; keep only its worst case.
closest = vox_centroids @ gallery_centroids.T
best = closest.max(axis=1)
matched = best >= TAU

print(f"VoxCeleb2 identities checked:  {len(vox_names):,}")
print(f"at or above tau={TAU:.3f}:      {matched.sum():,}  ({matched.mean():.1%})")
print(f"max similarity overall:        {best.max():.3f}")
print("\nclosest matches found:")
for index in np.argsort(-best)[:8]:
    flag = "DROP" if matched[index] else "keep"
    print(f"  {flag}  {vox_names[index]}  {best[index]:.3f}  ~  {gallery_names[closest[index].argmax()]}")

dropped = set(vox_names[matched])
structure = structure[~structure["identity"].isin(dropped)].reset_index(drop=True)
for identity in sorted(dropped):
    shutil.rmtree(FRAMES / identity, ignore_errors=True)

print(f"\nremoved {len(dropped):,} identities from the structure and from {FRAMES}")
print(f"training corpus: {structure['identity'].nunique():,} identities, {len(structure):,} frames,"
      f" none within cosine {TAU:.3f} of any gallery identity")

Embedding (ArcFace): 100%|██████████| 190560/190560 [11:59<00:00, 264.84frame/s]


VoxCeleb2 identities checked:  5,955
at or above tau=0.355:      210  (3.5%)
max similarity overall:        0.927

closest matches found:
  DROP  id06531  0.927  ~  Nicolas_Eyzaguirre
  DROP  id03804  0.914  ~  Jan_Peter_Balkenende
  DROP  id02343  0.914  ~  Ed_Rendell
  DROP  id07191  0.909  ~  Ralph_Klein
  DROP  id08999  0.908  ~  Whoopi_Goldberg
  DROP  id01042  0.898  ~  Bill_Belichick
  DROP  id05141  0.892  ~  Leticia_Van_de_Putte
  DROP  id00225  0.892  ~  Aleksander_Kwasniewski

removed 210 identities from the structure and from ../../data/voxceleb
training corpus: 5,745 identities, 3,329,036 frames, none within cosine 0.355 of any gallery identity


## 6. Save

`voxceleb.parquet` is the same shape as `ytfaces_wide.parquet`, so everything downstream takes it
unchanged — except that `identity` holds `id00012` rather than a name.

**Next**, to train on it:

1. Run `07_target_selection`'s logic over this structure to produce `voxceleb_targets.parquet` —
   embed, rank by margin, then pick the most frontal of each identity's shortlist. Step 5 already
   embedded 32 frames per identity, but target selection wants all of them, so it re-embeds.
2. In `06_facial` cell 2, read `voxceleb.parquet` and `voxceleb_targets.parquet`, and give the
   run a new `CHECKPOINT`: a different gallery means rank-1 is not comparable to the YouTube
   Faces runs.
3. Set `STRATIFY = True`. It was close to a no-op on YouTube Faces because 47% of identities had
   only one video; with ~24 it is the setting that spreads an item across sightings.

The staging directory can be deleted once the frames are written — it holds the raw clips and
nothing reads it again.

In [8]:
structure.to_parquet(DATA_PATH / "voxceleb.parquet", index=False)
print(f"wrote {DATA_PATH / 'voxceleb.parquet'}  ({len(structure):,} frames)")

# The loop clears STAGING as it goes, so only the archives are still holding disk.
archive_size = sum(f.stat().st_size for f in DOWNLOADS.rglob("*.zip"))
print(f"\narchives kept:  {archive_size / 1e9:.1f} GB in {DOWNLOADS}")
print(f"frames written: {sum(f.stat().st_size for f in FRAMES.rglob('*.jpg')) / 1e9:.1f} GB in {FRAMES}")
print("\nthe archives are only needed to re-extract at different settings; the frames are the dataset.")


wrote ../../data/voxceleb.parquet  (3,329,036 frames)

archives kept:  267.0 GB in ../../data/vox2_download
frames written: 44.0 GB in ../../data/voxceleb

the archives are only needed to re-extract at different settings; the frames are the dataset.
